In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import time


from __future__ import annotations
from yandex_cloud_ml_sdk import YCloudML


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [2]:
X_test = pd.read_csv('X_test_balanced.csv',index_col=0)

In [3]:
X_test.head(10)

,purpose
536723,взнос за июнь Карлова Татьяна;03/06/2025
15651,Ежемесячный членский взнос Мальцевой Н.П.за июль. НДС не облагается
599785,Ежемесячный членский взнос Кузнецовой Е.В.за август 2025г;01/08/2025
595334,"Выплата процентов по депозиту 8619750346.ПУ00 от 20.02.2025 за период с 21.02.2025 по 05.03.2025, без НДС"
495502,ЗА 23/05/2025;ПДО8;
391588,"Оплата по договору № ГФЦЗсп-005/23от 08.11.2023 г., пожертвование наосуществление проекта, 2-ой транш.Сумма 5998376-00. Без налога (НДС)."
147644,Перевод собственных средств. НДС не облагается
412493,Зачисление средств по операциям эквайринга. Мерчант №711000236317. Комиссия 2.10. НДС не облагается.
446092,Перевод собственных средств. НДС не облагается
294995,"Перевод с карты *2262, Благотворительное пожертвование на уставную деятельность. НДС не облагается"


In [ ]:
y_pred_ygpt_ft = pd.DataFrame(columns=["universal_category"])

sdk = YCloudML(
        folder_id="YOUR_FOLDER_ID",
        auth="YOUR_YANDEX_CLOUD_TOKEN",
    )

model = sdk.models.text_classifiers(
        "cls://YOUR_FOLDER_ID/yandexgpt-lite/rc@tamr9067f9ueb6v7f49gl"
    )

for index__, text in tqdm(X_test['purpose'].items(), total=len(X_test)):
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        try:
            result = model.run(str(text))  # приводим текст к строке
            best_label = max(result, key=lambda x: x.confidence).label
            y_pred_ygpt_ft.loc[index__, "universal_category"] = best_label
            break
        except Exception as e:
            attempts += 1
            print(f"Ошибка на index {index__}, попытка {attempts}/{max_attempts}: {e}")
            time.sleep(2)
    else:
        # если все попытки неудачные, присваиваем NaN
        y_pred_ygpt_ft.loc[index__, "universal_category"] = np.nan
    

100%|██████████| 900/900 [31:08<00:00,  2.08s/it]


In [5]:
y_pred_ygpt_ft.to_csv("y_pred_ygpt_ft.csv", index=True, header=["universal_category"])